# Guided tour of the financials factor pipeline

An end-to-end walk from **raw data → point-in-time panels → factors → validation &
selection** — as a thin visualisation layer over the precomputed `data/processed/`
artifacts (full universe, full history, built once by `scripts/build_processed.py`).
The heavy stages (lazy ingestion, PIT joins, factor computation, regularize →
neutralize) never run here; the notebook reads their outputs and focuses on the final
stage: turning the ~30-factor zoo into a **shortlisted, non-redundant set**.

The strategy universe is **financials (banks + insurers)**. Banks and insurers carry
mutually-exclusive metrics, so the pipeline is *sub-universe aware* throughout: factors
are partitioned by `Applicability`, and the cross-sectional steps never mix the two
sleeves. The structural-beta signals (`Beta~*`, registry `neutralize = False`) ride
along as betting-against-beta style factors with special handling (raw-return ICs,
lasso bypass).

> **Selection here is the in-sample demo** (`cutoff=None`: statistics use the full
> sample, so the selection itself has look-ahead). The production path is
> point-in-time: `scripts/select_features.py --schedule` runs the same
> `select_features` as of each re-selection cutoff with no data past the cutoff — see
> §4 and `DYNAMIC_SELECTION_PLAN.md`. Stage-by-stage schemas: `PIPELINE_REFERENCE.md`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import polars as pl
from plotnine import (ggplot, aes, geom_col, geom_line, geom_point, geom_hline,
                      geom_tile, geom_errorbar, coord_flip, facet_wrap,
                      scale_fill_manual, scale_fill_gradient2, scale_x_continuous,
                      labs, theme_bw, theme, element_text)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.factors.base import registry
from src.validation import single_factor as sf
from src.validation.selection import select_features

cfg = config.load(PROJECT_ROOT / "config.yaml")
CACHE = PROJECT_ROOT / cfg["data"]["cache"]

pl.enable_string_cache()   # Categorical stock_id joins across independently-read feathers
pl.Config.set_tbl_rows(35)

neu      = pl.read_ipc(CACHE / "neu_3m.feather")       # quarterly neutralized z-scores
returns  = pl.read_ipc(CACHE / "returns_3m.feather")   # periodic excess returns + meta
loadings = pl.read_ipc(CACHE / "loadings_3m.feather")  # structural design (MKT, beta_*, is_*)
events   = pl.read_ipc(CACHE / "delist_events.feather")

manifest = json.loads((CACHE / "manifest.json").read_text())
print(f"artifacts built {manifest['built_at']} @ {manifest['git_commit'][:8]} "
      f"| filters: {manifest['filters']}")
print(f"neu {neu.shape} | returns {returns.shape} | loadings {loadings.shape} "
      f"| delist events {events.height:,}")
print("periods:", neu["period"].min(), "->", neu["period"].max())

## 1. The precomputed inputs

| artifact | role here |
|---|---|
| `neu_3m` | quarterly neutralized z-scores (`regularize → neutralize`, full universe) — the selection inputs |
| `returns_3m` | per-(security, quarter) compounded excess returns, delisting-trimmed and terminal-settled; `ret_wins` (winsorized) is the estimation target, `ret_raw` the P&L one |
| `loadings_3m` | raw structural design (`MKT`, per-group `beta_*` / `is_*`) — the residualisation design for the ICs and the lasso target |
| `delist_events` | delisting classification of the tradeable set (survivorship handling, `DELISTING_HANDLING.md`) |

Two conventions to keep in mind: the `period` key (not `date`) defines a rebalance
cross-section, and the pre-built `neu_3m` is valid **only** for full-universe work —
sliced studies must rebuild from `scores_3m` / `loadings_3m`
(`data/processed/README.md`).

In [ ]:
cov = (returns.group_by("period", "industry")
       .agg(n=pl.col("stock_id").n_unique())
       .sort("period", "industry").to_pandas())
(ggplot(cov, aes("period", "n", fill="industry"))
 + geom_col(width=80)
 + labs(title="Tradeable universe per rebalance period", x="", y="# names", fill="")
 + theme_bw() + theme(figure_size=(10, 3.5), legend_position="top"))

In [ ]:
print(events.group_by("reason").len().sort("len", descending=True))
ev = (events.with_columns(year=pl.col("delist_date").dt.year())
      .group_by("year", "reason").len().sort("year").to_pandas())
(ggplot(ev, aes("year", "len", fill="reason"))
 + geom_col()
 + labs(title="Delisting events per year (survivorship handling)",
        x="", y="# events", fill="")
 + theme_bw() + theme(figure_size=(10, 3.5), legend_position="top"))

## 2. Neutralized cross-sections

`build_processed.py` finishes stage 4 for us: winsorize → median-impute → z-score
**within each factor's sub-universe** (a bank never receives an insurer's median), then
a per-`period` cross-sectional OLS residualises every style score against the
structural design (market + per-group country/industry betas and their baseline
dummies), re-standardized back to unit variance. The structural-beta signals pass
through un-residualised — they are exactly collinear with that design — and are only
restandardized.

The coverage map below shows the two structural patterns to expect: the ~2-year
rolling-beta **warm-up** on the left edge, and sector factors populating only their own
sleeve (though never infer sleeve membership from null patterns — factors like `B/P`
compute for both sectors).

In [ ]:
ID = {"stock_id", "date", "period", "industry"}
FACTORS = [c for c in neu.columns if c not in ID]
reg = registry()
STYLE = [f for f in FACTORS if reg[f].neutralize]
BETA  = [f for f in FACTORS if not reg[f].neutralize]
print(f"{len(FACTORS)} factors in the zoo: {len(STYLE)} style + {len(BETA)} "
      f"structural-beta signals {BETA}")

order = sorted(FACTORS, key=lambda f: (reg[f].applicability.value, f), reverse=True)
covf = (neu.unpivot(index=["period"], on=FACTORS, variable_name="factor", value_name="z")
        .group_by("period", "factor")
        .agg(coverage=pl.col("z").is_not_null().mean())
        .with_columns(factor=pl.col("factor").cast(pl.Enum(order)))
        .to_pandas())
(ggplot(covf, aes("period", "factor", fill="coverage"))
 + geom_tile()
 + labs(title="Factor coverage (share of the panel non-null, per rebalance period)",
        subtitle="rolling-beta warm-up on the left; sector factors cover only their own sleeve",
        x="", y="", fill="coverage")
 + theme_bw()
 + theme(figure_size=(11, 8), axis_text_y=element_text(size=7)))

## 3. Validation & selection — the three nested filters (in-sample demo)

One call to `src.validation.selection.select_features` replaces the old cell-by-cell
flow. Each step keeps fewer factors than the last (all thresholds from `config.yaml`'s
`validation:` block):

1. **Single-factor gate** — keep a factor iff `|IC-IR| ≥ 0.2` at lag 1 **or** pooled
   Fama-MacBeth `|t| ≥ 1`. Style-factor ICs correlate against loadings-residualised
   forward returns; the `Beta~*` signals against **raw** returns (residualising against
   the very loadings they are would annihilate them by construction).
2. **Redundancy** — correlation clustering **on the survivors only** (time-averaged
   `|ρ| > 0.4`); each cluster is represented by its largest-`|FM coefficient|` member
   (`cluster_representative: fm_gradient`).
3. **Parsimony** — `LassoCV` over the style representatives only (CV folds grouped by
   period); `Beta~*` representatives bypass the lasso and go straight to the shortlist.

`cutoff=None` = full sample: an honest demo of the machinery, not a tradable selection.

In [ ]:
train_start = cfg["backtest"]["schedule"]["train_start"]
sel = select_features(neu, returns, loadings, cfg, cutoff=None, train_start=train_start)

THR_IR = float(cfg["validation"]["ic"]["ir_shortlist_threshold"])
THR_T  = float(cfg["validation"]["fama_macbeth"]["t_stat_threshold"])
n_gate = sel.scorecard.filter(pl.col("single_pass")).height
multi = [c for c in sel.clusters if len(c) > 1]
print(f"single-factor gate (|IR| >= {THR_IR} or |FM t| >= {THR_T}): "
      f"{n_gate}/{len(FACTORS)} factors pass")
print(f"clusters with >1 member: {len(multi)} -> {len(sel.representatives)} representatives")
print(f"lasso survivors (style representatives only): {sorted(sel.lasso_survivors)}")
print(f"\nFINAL SHORTLIST ({len(sel.shortlist)}): {sel.shortlist}")

### 3a. IC-IR and IC decay

Rank IC = per-period Spearman correlation between the neutralized exposure and the
forward quarterly return (style factors: loadings-residualised target; `Beta~*`: raw
target). Its mean/std over periods is the **IC-IR**; the gate keeps `|IR| ≥ 0.2` at
lag 1. The longer horizons are diagnostics only (IC decay — resolved decision 5).

In [ ]:
irp = (sel.scorecard.select("factor", "ir_lag1", "ir_pass")
       .drop_nulls("ir_lag1").to_pandas())
(ggplot(irp, aes("reorder(factor, ir_lag1)", "ir_lag1", fill="ir_pass"))
 + geom_col() + coord_flip()
 + geom_hline(yintercept=[-THR_IR, THR_IR], linetype="dashed")
 + labs(title="Lag-1 IC-IR by factor", x="", y="IC-IR", fill=f"|IR| >= {THR_IR}")
 + theme_bw() + theme(figure_size=(8, 7)))

In [ ]:
decay = (sel.ic.filter(pl.col("factor").is_in(sel.shortlist))
         .group_by("factor", "lag").agg(ic=pl.col("ic").mean())
         .sort("factor", "lag").to_pandas())
(ggplot(decay, aes("lag", "ic", color="factor"))
 + geom_line() + geom_point()
 + labs(title="IC decay of the shortlisted factors", x="horizon (quarters)", y="mean IC")
 + theme_bw())

### 3b. Pooled Fama-MacBeth premia

Per-period cross-sectional regressions of forward returns on the dense factor set,
aggregated with Newey-West t-stats. The **pooled architecture** tests each factor
exactly once, on the universe it applies to: all-financials factors on the pooled
cross-section (sector factors riding along zero-filled off-sector as controls), bank /
insurance factors within their sleeve (all-financials factors as controls). `fm_sub` on
the scorecard records which run (`all` / `bank` / `insurance`) tested each factor.

In [ ]:
fmp = (sel.scorecard.select("factor", "fm_t", "fm_sub", "fm_pass")
       .drop_nulls("fm_t").to_pandas())
(ggplot(fmp, aes("reorder(factor, fm_t)", "fm_t", fill="fm_sub"))
 + geom_col() + coord_flip()
 + geom_hline(yintercept=[-THR_T, THR_T], linetype="dashed")
 + labs(title="Pooled Fama-MacBeth premia (Newey-West t)",
        x="", y="t-stat", fill="tested on")
 + theme_bw() + theme(figure_size=(8, 7)))

In [ ]:
fmc = (sel.fm.filter(pl.col("factor") != "const")
       .sort(pl.col("t_stat").abs(), descending=True, nulls_last=True)
       .unique("factor", keep="first")
       .drop_nulls("mean_coef")
       .with_columns(lo=pl.col("mean_coef") - pl.col("nw_se"),
                     hi=pl.col("mean_coef") + pl.col("nw_se"))
       .to_pandas())
(ggplot(fmc, aes("reorder(factor, mean_coef)", "mean_coef", color="sub_universe"))
 + geom_hline(yintercept=0, linetype="dashed", alpha=0.6)
 + geom_errorbar(aes(ymin="lo", ymax="hi"), width=0.2)
 + geom_point(size=2)
 + coord_flip()
 + labs(title="Fama-MacBeth factor premia (point estimate +/- 1 NW s.e.)",
        x="", y="mean coefficient", color="tested on")
 + theme_bw() + theme(figure_size=(8, 8)))

### 3c. Quantile return profiling

Rank IC is a *linear* rank measure and the FM premium a *linear* slope, so a factor can
clear the gate yet respond **non-linearly** — only the tails pay, or the middle humps.
`quantile_returns` buckets each factor's neutralized exposure into deciles per rebalance
period and averages each decile portfolio's forward return over periods. A clean linear
factor traces a monotone ramp from decile 1 (lowest exposure) to decile N (highest); a
kink or tails-only spike flags non-linearity the gate statistics would miss. Same
split-target treatment as the ICs: style factors profile a residualised return, `Beta~*`
signals the raw one. Highlighted panels = the **single-factor gate survivors** (pass
|IC-IR| *or* |FM t| — the set entering redundancy clustering below); survivors sit on
top, non-survivors below, each group ordered by |IC-IR|.

In [ ]:
QN = int(cfg["validation"]["quantiles"])
qr = pl.concat([
    sf.quantile_returns(neu.select("stock_id", "date", *STYLE), returns,
                        n_quantiles=QN, nonstyle_exposures=loadings,
                        target_col="ret_wins", delist_events=None),
    sf.quantile_returns(neu.select("stock_id", "date", *BETA), returns,
                        n_quantiles=QN, nonstyle_exposures=None,
                        target_col="ret_wins", delist_events=None),
])
gates = sel.scorecard.select("factor", "single_pass", "ir_lag1")
order = (gates.sort(pl.col("single_pass"), pl.col("ir_lag1").abs(),
                    descending=[True, True], nulls_last=True)["factor"].to_list())
order += [f for f in qr["factor"].unique() if f not in order]
survivors = gates.filter(pl.col("single_pass"))["factor"].to_list()
qrp = (qr.with_columns(
           status=pl.when(pl.col("factor").is_in(survivors))
                    .then(pl.lit("gate survivor")).otherwise(pl.lit("gate fail")),
           ret_pct=pl.col("mean_ret") * 100,
           factor=pl.col("factor").cast(pl.Enum(order)))
       .to_pandas())
(ggplot(qrp, aes("quantile", "ret_pct", fill="status"))
 + geom_col()
 + geom_hline(yintercept=0, size=0.3)
 + facet_wrap("~factor", ncol=4, scales="free_y")
 + scale_fill_manual(values={"gate survivor": "#2c7fb8", "gate fail": "#c9c9c9"})
 + scale_x_continuous(breaks=[1, QN])
 + labs(title=f"Decile forward-return profiles ({QN} buckets, one quarter ahead)",
        subtitle="bucket 1 = lowest ... bucket N = highest exposure; "
                 "gate survivors on top, |IC-IR|-ordered within each group",
        x="exposure decile", y="mean quarterly excess return (%)", fill="")
 + theme_bw()
 + theme(figure_size=(14, 12), subplots_adjust={"hspace": 0.55, "wspace": 0.3},
         strip_text=element_text(size=8), axis_text=element_text(size=6),
         legend_position="top"))

### 3d. Redundancy — correlation clustering on the gate survivors

Factor pairs whose time-averaged cross-sectional correlation exceeds the threshold are
grouped into clusters (connected components); each cluster keeps its
largest-`|FM coefficient|` member (`fm_gradient` — the factor with the steepest
premium, not the best IR). Clustering runs on the **single-factor survivors only**, so
the heatmap below is the survivor set.

In [ ]:
corr_thr = float(cfg["validation"]["redundancy"]["correlation_threshold"])
for i, cluster in enumerate(sel.clusters):
    if len(cluster) > 1:
        rep = next(f for f in cluster if f in sel.representatives)
        print(f"cluster {i}: keep {rep!r} (largest |FM coef|), "
              f"drop {[f for f in cluster if f != rep]}")
print(f"\nflagged pairs (time-averaged |rho| > {corr_thr}): {sel.flagged.height}")

sym = pl.concat([
    sel.pairs,
    sel.pairs.rename({"factor_a": "factor_b", "factor_b": "factor_a"})
    .select("factor_a", "factor_b", "rho"),
]).to_pandas()
(ggplot(sym, aes("factor_a", "factor_b", fill="rho"))
 + geom_tile()
 + scale_fill_gradient2(low="#d95f02", mid="white", high="#2c7fb8", limits=(-1, 1))
 + labs(title="Average cross-sectional correlation of the gate survivors",
        x="", y="", fill="rho")
 + theme_bw()
 + theme(figure_size=(8, 7), axis_text_x=element_text(rotation=45, hjust=1)))

### 3e. Parsimony lasso & the final scorecard

The cluster representatives enter a cross-validated lasso (per sub-universe, CV folds
grouped by period, target residualised against the loadings); survivors form the final
shortlist. Scorecard legend: `lasso` is `true`/`false` when the factor entered the
lasso and survived/dropped, `null` when it never entered (failed an earlier step, or a
`Beta~*` signal that bypasses it — such a signal is `selected` iff it passed the gate
and is a cluster representative).

In [ ]:
print(f"FINAL SHORTLIST ({len(sel.shortlist)}): {sel.shortlist}")
sel.scorecard.drop("cutoff")

## 4. Point-in-time selection — the production path

The demo above sees the full sample. The dynamic backtest instead re-runs
`select_features` **as of each re-selection cutoff** (every 2 years, 2005Q4 → 2023Q4),
slicing every input to `period ≤ cutoff` *before* the forward shift so no post-cutoff
observation enters any statistic:

    conda run -n ip2 python scripts/select_features.py --schedule

writes per-cutoff scorecards and `shortlists.json` to `data/processed/selection/`. The
factor set genuinely evolves — the selection-dynamics section of
`backtest_tour.ipynb` quantifies it.

In [ ]:
shortlists = json.loads((CACHE / "selection" / "shortlists.json").read_text())
pl.Config.set_fmt_str_lengths(160)
pl.DataFrame({
    "cutoff": list(shortlists),
    "n": [len(v) for v in shortlists.values()],
    "shortlist": [", ".join(v) for v in shortlists.values()],
})

## Caveats & pointers

* **The selection above is in-sample** — one full-sample demo call. Any traded use goes
  through the PIT path (§4); the leak test in `tests/test_selection.py` proves the
  cutoff discipline (corrupting all post-cutoff data leaves the shortlist
  bit-identical).
* **Never slice `neu_3m`** — its cross-sectional statistics are full-universe; sliced
  studies must rebuild from `scores_3m` / `loadings_3m` (`data/processed/README.md`).
* **Warm-up** — rolling structural betas need ~2 years of history, so the neutralized
  panel is effectively valid from ~2000 (= the backtest schedule's `train_start`).
* All thresholds (IR gate, FM gate, correlation cut-off, quantile count, lasso
  settings) come from `config.yaml`'s `validation:` block.
* Next stop: `backtest_tour.ipynb` — the dynamic walk-forward backtest that re-runs
  this selection point-in-time every two years and trades the result.